# Redispatch probability — day-ahead demo

Pick a **town** and a **date** in the test set (2026-01-01 .. 2026-03-30). The notebook returns hour-by-hour probabilities for the three horizons, overlays them on a chart, places the town on a map of all 175 SHN substations, and explains the model's decision at the most-confident hour using LightGBM's per-feature contribution scores (SHAP values, no extra deps).

Sections
1. Setup
2. Inventory of available towns and dates
3. Pick town + date → predict
4. Probability curves
5. Map with chosen town highlighted
6. Feature attribution (top contributors at the peak hour)

## 1. Setup

In [1]:
import sys, json
from pathlib import Path

# Use relative path resolution instead of hardcoded absolute path
ROOT = Path.cwd()
while ROOT.name not in ('Redispatch', ''):
    ROOT = ROOT.parent
if ROOT.name == '':
    raise RuntimeError('Could not find project root directory')

sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium

import predict as P
from predict import predict_day, plot_day

# warm the caches once so subsequent calls are instant
P._load()
print('models:', list(P._MODELS.keys()))
print('feature count:', len(P._FCOLS))

ModuleNotFoundError: No module named 'predict'

## 2. Inventory — what can I pick?

Lists the test-window date range and the top-25 busiest towns by redispatch hours in the test window, so picking a likely-active candidate is one click.

In [ ]:
feats = P._FEATURES
test = feats[feats['ts'] >= pd.Timestamp('2026-01-01')]
print(f'test rows : {len(test):,}')
print(f'date range: {test["ts"].min()} -> {test["ts"].max()}')
print(f'n towns   : {test["town"].nunique()}')

busiest = (test.groupby('town')['is_active'].sum()
                .sort_values(ascending=False).head(25))
print('\nTop-25 busiest towns in test window (active hours):')
print(busiest.to_string())

## 3. Pick town + date → predict

In [ ]:
TOWN = 'Linden'         # <-- edit me
DATE = '2026-01-19'    # <-- edit me (YYYY-MM-DD, must be in test window)

df = predict_day(TOWN, DATE)
df

## 4. Probability curves

Lines = predicted P(active in next 1h / 6h / 24h). Hollow circles = the actual `y_*` labels (where 1 = redispatch did happen). Where lines pass through circles, the model called it; where circles are floating above flat low lines, the model missed it; where lines are high but no circle, false alarm.

In [ ]:
plot_day(df, TOWN, DATE)

actual_24h = bool(df['y_24h'].max()) if 'y_24h' in df.columns else None
peak_p1h = df['p_1h'].max()
print(f'peak p_1h on this day: {peak_p1h:.3f}  (at {df.loc[df["p_1h"].idxmax(), "ts"]})')
print(f'mean p_24h:           {df["p_24h"].mean():.3f}')
print(f'actual y_24h on day:  {actual_24h}')

## 5. Map — where is this town in the SHN grid?

In [ ]:
geo  = pd.read_parquet(ROOT / 'data' / 'external' / 'towns_geo.parquet').dropna(subset=['lat','lon'])
wide = pd.read_parquet(ROOT / 'data' / 'processed' / 'ts_15min_wide.parquet')

intensity = (wide > 0).sum().rename('active_slots').reset_index()
intensity.columns = ['town', 'active_slots']
g = geo.merge(intensity, on='town')

m = folium.Map(location=[54.3, 9.7], zoom_start=8, tiles='cartodbpositron')
maxv = g['active_slots'].max()
for _, r in g.iterrows():
    is_pick = (r['town'] == TOWN)
    folium.CircleMarker(
        location=(r.lat, r.lon),
        radius=(20 if is_pick else 3 + 18 * (r.active_slots / maxv)),
        popup=f"{r.town}: {int(r.active_slots):,} active 15-min slots",
        color=('blue' if is_pick else 'crimson'),
        weight=(3 if is_pick else 1),
        fill=True,
        fill_color=('blue' if is_pick else 'crimson'),
        fill_opacity=(0.9 if is_pick else 0.4),
    ).add_to(m)

# dashed ring around the pick to make it pop
pick = g[g['town'] == TOWN]
if not pick.empty:
    folium.Circle(
        location=(pick.iloc[0].lat, pick.iloc[0].lon),
        radius=8000, color='blue', weight=2, fill=False, dash_array='6',
    ).add_to(m)
    folium.Marker(
        location=(pick.iloc[0].lat, pick.iloc[0].lon),
        icon=folium.DivIcon(html=f'<div style="font-weight:bold;color:blue">{TOWN}</div>'),
    ).add_to(m)

m

## 6. Why? — feature attribution at the peak hour

Uses LightGBM's `pred_contrib=True` (TreeSHAP) to decompose the predicted log-odds at the peak hour into per-feature contributions. Positive contribution = pushed probability up; negative = pulled it down. Top 12 by absolute magnitude.

Town one-hots are aggregated into a single bar (`town_*` collectively), so the chart isn't dominated by 175 nearly-zero bars.

In [ ]:
HZ = 'y_24h'   # which model to explain — edit if you'd like

models, fcols = P._MODELS, P._FCOLS
X_day = P._FEATURES[(P._FEATURES['town'] == TOWN) &
                    (P._FEATURES['ts'].dt.normalize() == pd.Timestamp(DATE))][fcols].copy()

p_col = {'y_1h':'p_1h', 'y_6h':'p_6h', 'y_24h':'p_24h'}[HZ]
peak_idx_local = int(np.argmax(df[p_col].values))
peak_ts        = df.iloc[peak_idx_local]['ts']
peak_p         = df.iloc[peak_idx_local][p_col]

x_peak = X_day.iloc[[peak_idx_local]].values
contrib = models[HZ].predict(x_peak, pred_contrib=True)[0]   # last entry is bias
bias, contrib = contrib[-1], contrib[:-1]

ctr = pd.DataFrame({'feature': fcols, 'contribution': contrib})

# collapse town_* one-hots into one row
is_town = ctr['feature'].str.startswith('town_')
town_total = ctr.loc[is_town, 'contribution'].sum()
ctr = pd.concat([
    ctr[~is_town],
    pd.DataFrame({'feature': ['town_* (175 cols)'], 'contribution': [town_total]}),
], ignore_index=True)
ctr['abs'] = ctr['contribution'].abs()
ctr = ctr.sort_values('abs', ascending=False).head(12).iloc[::-1]

fig, ax = plt.subplots(figsize=(8.5, 5))
colors = ['#2ca02c' if v > 0 else '#d62728' for v in ctr['contribution']]
ax.barh(ctr['feature'], ctr['contribution'], color=colors, edgecolor='k', linewidth=0.4)
ax.axvline(0, color='k', lw=0.6)
ax.set_xlabel('contribution to log-odds (green = pushed P up, red = pulled P down)')
ax.set_title(f'{TOWN} | {peak_ts}  |  {HZ}={peak_p:.3f}\n'
             f'(bias={bias:+.2f}, sum={contrib.sum():+.2f})')
fig.tight_layout()
plt.show()

# also show the raw feature values at that hour for the top contributors
show_cols = [f for f in ctr['feature'] if not f.startswith('town_')]
row = X_day.iloc[peak_idx_local]
print('\nfeature values at peak hour:')
for f in reversed(show_cols):
    if f in row:
        print(f'  {f:<28s} = {row[f]:.3f}')

## How to read this

* **High `p_24h`, low `p_1h`** through the day → model is confident *something* will happen tomorrow but uncertain *when*. Common output for windy days.
* **`p_1h` spike with no `p_6h` / `p_24h` rise** → unusual, would suggest a brief, localized event the longer-horizon models smoothed out.
* **Green bars dominated by `wind_onshore` + `wx_wind_speed_100m`** → classic wind-curtailment signature.
* **Big positive `active_7d_lag24` contribution** → persistence kicking in (this town has been busy recently).
* **Big positive `lat`** → north-coast geographic prior taking over (the model learned wind farms cluster north).

If you want to see a clearly-firing day, try `TOWN='Reinsbüttel'` or `'Linden'` on a date near a January storm: `'2026-01-19'` or `'2026-02-08'`.